# Does the ranking invert when nobody gives up?

`carla-sustained-load-capacity-12.ipynb` pushed three towns to **68.8% seat
occupancy** — past the ~60% line at which the synthetic map says
demand-responsive routing stops paying — and found **no inversion anywhere**.
Reordering led fixed route at every rung in every town.

That result is real but it answers a narrower question than it appears to.
It was measured on **coverage in the impatient regime**, and the synthetic
evidence splits sharply by regime:

| regime | what happens above ~60% occupancy | measured on |
|---|---|---|
| **impatient** (patience 900 s) | margin decays 0.21 -> -0.02; the best arm never crosses | served fraction |
| **patient** (nobody gives up) | Reordering goes from 150 s **better** than fixed route to **5,090 s worse** | mean wait |

**The CARLA sweep tested the cell where the synthetic inversion is weakest,
and the cell where it is dramatic has never been run on real geometry.** This
notebook runs it.

## Why the patient regime is worth an evening even though it is not realistic

It is not the operating regime anyone would deploy — real passengers abandon,
and the impatient sweep is the honest picture of service. Two reasons to
measure it anyway:

1. **It is the only regime in which wait time means anything.** Wait and ride
   are computed over *fulfilled* requests. When passengers abandon, a policy
   that gives up on its hard riders posts a better mean wait, and when the
   service window equals patience nobody who waited long is left in the
   sample at all. With no give-up, **every rider appears in the mean**, so
   wait becomes an uncensored measurement rather than a survivor statistic.
2. **It is where the synthetic map put the inversion.** Claiming the scope
   condition does or does not transfer to CARLA means testing the same thing
   the synthetic map tested, not its neighbour.

So this is a diagnostic, not a service claim. Nothing here changes what to
recommend for a deployment; it changes what we are entitled to say about
*why* the recommendation has a ceiling.

## One variable

Every constant below is copied from the impatient sweep — towns, capacity,
span, rungs, seeds, service window, NPC count, dwell, steering tag.
**`max_wait_s` alone changes, from 750 s to `None`.** The two notebooks are
therefore a matched pair, and section 7 differences them directly out of the
store rather than transcribing numbers between notebooks.

!!! warning "What `max_wait_s=None` changes about the run"
    Nothing cancels, so **`served_fraction` is 1.0 by construction** for any
    trial that finishes. The 80% gate is meaningless here and is not drawn.
    It also means coverage doubles as a truncation detector: a run below
    100% did not finish, and section 5 exists to catch exactly that.

    It also removes the tail bound. In the impatient sweep an unserved rider
    abandoned after 750 s; here the bus must serve everyone, so the drain
    after demand stops is limited only by `MAX_TIME`.

## Configuration

Two towns rather than the sibling's three, and TAU first. TAU is the
project's own map, the cheapest to drive, **and the town where routing's
advantage was largest** (50 points at the easiest rung) — so it is the least
likely place to see an inversion and the most decisive if one appears.
Town05 is the opposite end: the hardest town and the smallest advantage
(10.7 points at the top rung), the most likely to invert.

Town03 sits between them and is dropped for cost. Adding it back is one
entry in `TOWNS`.

**Nothing here is reused from the impatient sweep.** `max_wait_s` is part of
a run's identity, so all 36 trials are new, including the anchor rung.

In [1]:
# --- Towns ---------------------------------------------------------------
# TAU first: cheapest to drive, the project's own map, and the town where
# routing's advantage was largest in the impatient sweep -- so it is the
# least likely place for an inversion and the most decisive if one shows.
# Town05 is the other end: hardest town, smallest advantage. Town03 sits
# between them and is dropped for cost; add it back here.
TOWNS = ["TAU", "Town05"]

# --- Copied from carla-sustained-load-capacity-12.ipynb ------------------
# Every one of these matches that notebook field for field. They are the
# controlled half of a matched pair, and changing any of them breaks the
# comparison in section 7.
NPC_VEHICLES = 20
CAPACITY = 12
WINDOW = 750.0
DWELL_S = 10.0
FIXED_DELTA_S = 0.05
ARRIVAL_SPAN_S = 2700.0
DEMAND_INTERVALS = [90.0, 45.0, 30.0]
SEEDS = list(range(3))

# --- The one variable ----------------------------------------------------
# None means a request never abandons: it stays PENDING until the bus
# reaches it. This is the whole experiment.
PATIENCE = None

# What the sibling ran, for section 7's comparison. Named rather than
# written inline so the two store filters cannot drift apart.
IMPATIENT_PATIENCE = 750.0

# --- The runaway guard ---------------------------------------------------
# Cut from the sibling's 40,000 s. With no give-up the drain after demand
# stops is bounded only by this number, and a trial that cannot finish
# would otherwise cost most of an hour on a slow town. 15,000 s is 5.6x the
# arrival span and ~2.5x the expected drain; section 5 treats any run that
# reaches it as invalid rather than slow.
MAX_TIME = 15000.0

CARLA_HOST, CARLA_PORT = "localhost", 2000

### What this costs, and the one way it can run away

Wall time tracks simulated time, and simulated time here is the 2700 s span
plus however long the bus needs to finish everyone still waiting. In the
impatient sweep that tail was bounded by patience. **It is not bounded here**,
which is the one thing about this notebook that can go wrong expensively.

The estimate below scales the sibling's measured per-trial minutes by the
expected drain: at the top rung roughly a third of the riders went unserved
in the impatient sweep, and at ~1.3 fulfilled/min serving them costs about
another 1,400 s of simulation.

`MAX_TIME` is cut from the sibling's 40,000 s to **15,000 s** for this
reason. That is 5.6x the arrival span and about 2.5x the expected drain —
generous enough that a healthy run never sees it, tight enough that a trial
which cannot finish costs minutes instead of an hour. A run that hits it is
not a slow run, it is a stuck one, and section 5 reports it as invalid
rather than averaging it in.

In [2]:
def requests_for(interval_s: float) -> int:
    """
    How many requests fill the fixed arrival span at a given rate.

    :param interval_s: Demand interval in seconds.
    :return: Requests to emit, rounded to a whole rider.
    """
    return round(ARRIVAL_SPAN_S / interval_s)


# Resolved once so the runner, the store filter and the tables cannot
# disagree about what a rung is.
RUNG_REQUESTS = {i: requests_for(i) for i in DEMAND_INTERVALS}

# Simulated seconds per wall minute, from the impatient sweep's measured
# per-trial times against its ~3,600 s runs. Expressed as a rate rather
# than as minutes per trial because the runs here are longer, and the rate
# is the part that carries over.
SIM_S_PER_WALL_MIN = {"TAU": 2130.0, "Town03": 683.0, "Town05": 589.0,
                      "Town01": 692.0, "Town02": 839.0}
DEFAULT_SIM_S_PER_WALL_MIN = 700.0

# The impatient sweep left roughly a third of the riders unserved at the
# top rung. Here the bus has to fetch them too, at ~1.3 fulfilled/min --
# about 1,400 s more simulation on top of a ~4,000 s run. ETA only.
EXPECTED_SIM_S = 5500.0


def wall_minutes(town: str, sim_s: float = EXPECTED_SIM_S) -> float:
    """
    Estimated wall minutes to simulate ``sim_s`` seconds in one town.

    :param town: Town the trial runs in.
    :param sim_s: Simulated seconds to price.
    :return: Minutes.
    """
    rate = SIM_S_PER_WALL_MIN.get(town, DEFAULT_SIM_S_PER_WALL_MIN)
    return sim_s / rate


print(f"arrival span held at {ARRIVAL_SPAN_S:.0f} s for every rung")
print("patience: none -- no request is ever cancelled\n")
for interval in DEMAND_INTERVALS:
    print(f"  {60 / interval:>4.2f}/min  ->  {RUNG_REQUESTS[interval]:>3} "
          f"requests  (interval {interval:.0f} s)")

per_town = 2 * len(SEEDS) * len(DEMAND_INTERVALS)
print(f"\n{len(TOWNS)} towns x 2 policies x {len(SEEDS)} seeds x "
      f"{len(DEMAND_INTERVALS)} rungs = {len(TOWNS) * per_town} trials, "
      "all of them new")
print("(max_wait_s is part of a run's identity, so nothing is reused from "
      "the impatient sweep)\n")
for town in TOWNS:
    print(f"  {town:<8} {per_town:>2} trials  "
          f"~{wall_minutes(town) * per_town / 60:>4.1f} h expected   "
          f"{wall_minutes(town, MAX_TIME) * per_town / 60:>5.1f} h if every "
          "one truncates")
print(f"\n  {'total':<8} {len(TOWNS) * per_town:>2} trials  "
      f"~{sum(wall_minutes(t) for t in TOWNS) * per_town / 60:>4.1f} h "
      "expected")
print("\nStopping after TAU still answers the question, on one map.")

arrival span held at 2700 s for every rung
patience: none -- no request is ever cancelled

  0.67/min  ->   30 requests  (interval 90 s)
  1.33/min  ->   60 requests  (interval 45 s)
  2.00/min  ->   90 requests  (interval 30 s)

2 towns x 2 policies x 3 seeds x 3 rungs = 36 trials, all of them new
(max_wait_s is part of a run's identity, so nothing is reused from the impatient sweep)

  TAU      18 trials  ~ 0.8 h expected     2.1 h if every one truncates
  Town05   18 trials  ~ 2.8 h expected     7.6 h if every one truncates

  total    36 trials  ~ 3.6 h expected

Stopping after TAU still answers the question, on one map.


In [3]:
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from tqdm.auto import tqdm

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.experiment import run_campaign
from src.experiment.store import DEFAULT_RUNS_DIR, load_runs
from src.map.carla_stops import stops_path
from src.map.carla_topology import topology_path
from src.map.CarlaMapProvider import CarlaMapProvider
from src.simulation.CarlaBackend import (
    CarlaBackend, speed_limits_from, stop_positions_from,
)
from src.simulation.carla_world import (
    GENERATED_SPEED_CEILING_KPH, is_generated,
)

from IPython.display import Markdown, display

# The campaign's palette, so all four CARLA notebooks are read together.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

POLICY_COLOUR = {"Fixed route": SERIES[0], "Reordering W750": SERIES[2]}
POLICY_LABELS = ["Fixed route", "Reordering W750"]
POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Reordering W750",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW)),
]

# demand_interval_s and max_requests are absent: both are swept and both
# are supplied per rung. max_wait_s=None is the experiment.
BASE_TRIAL = dict(
    capacity=CAPACITY, max_wait_s=PATIENCE, dwell_s=DWELL_S,
    max_time=MAX_TIME,
)

# The occupancy at which the synthetic map's ranking inverts. Drawn as a
# reference line, never assumed by the analysis.
INVERSION_OCCUPANCY = 60.0

## 1. Are the towns prepared?  *(no server needed)*

Cheap, and it fails loudly rather than letting the sweep skip a town and
report one. A `traffic_efficiency` of 1.0 means uncalibrated: run
`carla-campaign-towns.ipynb` section 3 first.

In [4]:
def speed_limit_tag(town: str) -> str:
    """
    How this town's speed limits were derived, for the run identity.

    A shipped town publishes type-274 landmarks; a generated one has none,
    so its limits come from the .xodr's per-lane elements and are capped.
    The two are different procedures and must not pool -- which is also why
    this is a function rather than a literal: hardcoding the shipped town's
    tag in a filter silently drops every generated town's runs.

    :param town: Town name.
    :return: The identity tag for that town's speed-limit source.
    """
    if is_generated(town):
        return f"xodr-capped-{int(GENERATED_SPEED_CEILING_KPH)}-v1"
    return "landmark-harmonic-v1"


def readiness(town: str) -> dict:
    """
    Report whether one town can be swept.

    :param town: Town name.
    :return: A row describing what is present and what is not.
    """
    has_topology = topology_path(town).exists()
    has_stops = stops_path(town).exists()
    efficiency, stops, note = None, None, ""
    if has_topology and has_stops:
        try:
            provider = CarlaMapProvider(town)
            efficiency = provider.traffic_efficiency()
            stops = len(provider.get_stops())
            problems = provider.validate()
            note = "OK" if not problems else f"{len(problems)} map problems"
        except Exception as error:              # noqa: BLE001
            note = f"failed to load: {error!r}"
    return {
        "town": town,
        "topology": "yes" if has_topology else "MISSING",
        "stops": stops if stops is not None else (
            "yes" if has_stops else "MISSING"),
        "traffic_efficiency": efficiency if efficiency is not None else "-",
        "speed_limits": speed_limit_tag(town),
        "ready": "yes" if efficiency not in (None, 1.0) else "no",
        "note": note,
    }


display(pd.DataFrame([readiness(t) for t in TOWNS]).set_index("town"))

READY_TOWNS = [
    t for t in TOWNS
    if stops_path(t).exists() and topology_path(t).exists()
    and CarlaMapProvider(t).traffic_efficiency() != 1.0
]
print(f"ready to sweep: {READY_TOWNS}")
missing = [t for t in TOWNS if t not in READY_TOWNS]
if missing:
    display(Markdown(
        f"> **Not sweeping: {', '.join(missing)}.** Uncalibrated or missing "
        "an artifact -- prepare it in `carla-campaign-towns.ipynb` "
        "sections 2-3 first."))

,topology,stops,traffic_efficiency,speed_limits,ready,note
town,,,,,,
TAU,yes,25,0.7032,xodr-capped-50-v1,yes,34 map problems
Town05,yes,15,0.3309,landmark-harmonic-v1,yes,OK


ready to sweep: ['TAU', 'Town05']


## 2. Run the sweep  🖥 *needs a live server*

**The CARLA server is shared — check nobody else is using it before
starting.** Sequential: `CarlaBackend` sets `parallel_safe = False`.

Town-major with TAU first, so an interrupted evening leaves one town
complete rather than two half-done. Everything lands in the shared results
store and anything already recorded is skipped, so a resumed run picks up
where it stopped.

One `run_campaign` call per trial, so a CARLA hiccup costs one trial rather
than the night.

In [ ]:
def world_for(town: str, provider: CarlaMapProvider) -> dict:
    """
    Build the store identity for one town's world.

    Identical to the impatient sweep's, field for field. The regime lives
    in the trial identity (``max_wait_s``), not here, which is what keeps
    the two sweeps' worlds comparable.

    :param town: Town name.
    :param provider: That town's loaded provider.
    :return: The world identity.
    """
    return {
        "town": town, "backend": "CarlaBackend",
        "npc_vehicles": NPC_VEHICLES, "fixed_delta_s": FIXED_DELTA_S,
        "no_rendering": True, "blueprint": "vehicle.volkswagen.t2_2021",
        "traffic_efficiency": provider.traffic_efficiency(),
        "speed_limits": speed_limit_tag(town),
        "steering": "direct-drive-v2",
        "routing": "time-lane-change-v1",
    }


def backend_factory_for(town: str, positions: dict, speed_for):
    """
    Build a per-seed world builder for one town.

    :param town: Town to drive in.
    :param positions: That town's stop coordinates.
    :param speed_for: That town's speed-limit lookup, so the planner prices
        legs in travel time exactly as TransitGraph does.
    :return: A function of seed returning a backend factory.
    """
    def for_seed(seed: int):
        def build(graph, start_stop_id: str) -> CarlaBackend:
            """Build the CARLA backend for one trial."""
            return CarlaBackend(
                stop_positions=positions, position=start_stop_id, town=town,
                speed_for=speed_for, lane_changes=True,
                host=CARLA_HOST, port=CARLA_PORT,
                fixed_delta_s=FIXED_DELTA_S, dwell_s=DWELL_S,
                npc_vehicles=NPC_VEHICLES, seed=seed, no_rendering=True,
                # CARLA results are not independent of server history: an
                # earlier campaign lost 7 Town05 runs to a back-to-back
                # trials artefact and had to be discarded.
                reload_world=True)
        return build
    return for_seed


RUN_SWEEP = True    # set False to skip; needs a live server

if RUN_SWEEP and not READY_TOWNS:
    display(Markdown("> **Nothing to run.** No town is prepared."))

if RUN_SWEEP and READY_TOWNS:
    failures, durations = [], []
    towns_bar = tqdm(READY_TOWNS, desc="towns", unit="town")
    for town in towns_bar:
        towns_bar.set_postfix_str(town)
        provider = CarlaMapProvider(town)
        positions = stop_positions_from(provider)
        world = world_for(town, provider)
        for_seed = backend_factory_for(
            town, positions, speed_limits_from(provider))
        expected = wall_minutes(town)

        rungs_bar = tqdm(DEMAND_INTERVALS, desc=f"  {town}", unit="rung",
                         leave=False)
        for interval in rungs_bar:
            count = RUNG_REQUESTS[interval]
            rungs_bar.set_postfix_str(
                f"{60 / interval:.2f}/min, {count} requests")
            trials_bar = tqdm(
                total=len(SEEDS) * len(POLICIES), unit="trial", leave=False,
                desc=f"    {60 / interval:.2f}/min")
            # Seed-major so both policies of a seed finish together: an
            # interrupted run then leaves paired data rather than an orphan.
            for seed in SEEDS:
                for label, factory in POLICIES:
                    started = time.perf_counter()
                    ran = 0
                    try:
                        outcome = run_campaign(
                            policies=[(label, factory)], seeds=[seed],
                            world=world, directory=DEFAULT_RUNS_DIR,
                            provider=provider, backend_factory_for=for_seed,
                            n_jobs=1, progress=False,
                            demand_interval_s=interval,
                            max_requests=count, **BASE_TRIAL)
                        ran = outcome.ran
                    except Exception as error:      # noqa: BLE001
                        # Broad on purpose: a CARLA failure surfaces as
                        # almost anything, and losing the rest is worse.
                        failures.append((town, interval, seed, label,
                                         repr(error)))
                        tqdm.write(f"  {town} {60 / interval:.2f}/min "
                                   f"seed {seed} {label} FAILED: {error!r}")

                    elapsed = (time.perf_counter() - started) / 60
                    # A skipped trial costs no time; counting it would make
                    # the ETA nonsense for the rest of a resumed run.
                    if ran:
                        durations.append(elapsed)
                    rolling = (statistics.fmean(durations[-5:]) if durations
                               else expected)
                    trials_bar.update(1)
                    trials_bar.set_postfix_str(
                        f"seed {seed} {label}"
                        + ("" if ran else " (already recorded)")
                        + f" | rolling {rolling:.1f} min/trial, "
                          f"expected {expected:.1f}")
            trials_bar.close()
        rungs_bar.close()

    planned = (len(READY_TOWNS) * len(DEMAND_INTERVALS) * len(SEEDS)
               * len(POLICIES))
    print(f"\ndone -- {len(durations)} trials run, "
          f"{planned - len(durations)} skipped or failed, "
          f"{len(failures)} failures")
    for town, interval, seed, label, error in failures:
        print(f"  {town} {60 / interval:.2f}/min seed {seed} {label}: {error}")
elif not RUN_SWEEP:
    print("sweep skipped -- set RUN_SWEEP = True with a server up")

towns:   0%|          | 0/2 [00:00<?, ?town/s]

  TAU:   0%|          | 0/3 [00:00<?, ?rung/s]

    0.67/min:   0%|          | 0/6 [00:00<?, ?trial/s]

    1.33/min:   0%|          | 0/6 [00:00<?, ?trial/s]

    2.00/min:   0%|          | 0/6 [00:00<?, ?trial/s]

## 3. Collect

In [ ]:
def collect(patience: float | None) -> pd.DataFrame:
    """
    Read back every run belonging to one regime of this sweep.

    Both regimes are read by this one function so that section 7 compares
    two store queries rather than a query against transcribed numbers --
    which is how the sibling notebook came to difference two capacities and
    label the result a volume cost.

    :param patience: ``max_wait_s`` the runs were filed under: ``None`` for
        this notebook's patient runs, 750.0 for the impatient sibling's.
    :return: One row per run.
    """
    records = [
        r for r in load_runs(DEFAULT_RUNS_DIR,
                             where={"identity.world.backend": "CarlaBackend"})
        if r.identity.world.get("town") in TOWNS
        and r.identity.world.get("npc_vehicles") == NPC_VEHICLES
        and r.identity.world.get("speed_limits") == speed_limit_tag(
            r.identity.world.get("town"))
        and r.identity.world.get("steering") == "direct-drive-v2"
        and r.identity.world.get("routing") == "time-lane-change-v1"
        and r.identity.trial.get("capacity") == CAPACITY
        and r.identity.trial.get("max_wait_s") == patience
        # The pairing that separates a sustained rung from the burst
        # sweep's 30-request runs at the same rate.
        and RUNG_REQUESTS.get(
            r.identity.trial.get("demand_interval_s")
        ) == r.identity.trial.get("max_requests")
        and r.identity.policy in POLICY_LABELS
        and r.identity.seed in SEEDS
    ]
    rows = []
    for r in records:
        interval = r.identity.trial["demand_interval_s"]
        sim_s = r.backend_stats.get("ticks", 0) * FIXED_DELTA_S
        rows.append({
            "town": r.identity.world["town"], "policy": r.identity.policy,
            "seed": r.identity.seed, "interval_s": interval,
            "offered_per_min": 60.0 / interval,
            "requests": r.identity.trial["max_requests"],
            "served": r.kpi.served_fraction * 100,
            "wait": r.kpi.mean_wait_time, "ride": r.kpi.mean_ride_time,
            "p90_wait": r.kpi.p90_wait_time,
            "throughput": r.kpi.throughput,
            "fulfilled": r.kpi.fulfilled_count,
            "cancelled": r.kpi.cancelled_count,
            "sim_s": sim_s,
            "distance_m": r.backend_stats.get("distance_m", 0.0),
            # Under direct drive a repath means the bus was physically
            # stuck; the red-light and queueing exemptions cover the
            # legitimate stops.
            "repaths": r.backend_stats.get("repaths", 0),
            "unroutable": r.backend_stats.get("unroutable_legs", 0),
        })
    return pd.DataFrame(rows)


frame = collect(PATIENCE)
HAVE_RUNS = not frame.empty

if not HAVE_RUNS:
    display(Markdown(
        "> **No runs yet.** Run section 2, or check that the world identity "
        "here matches what the runs were filed under."))
else:
    print(f"{len(frame)} runs, {frame['town'].nunique()} towns, "
          f"{frame['interval_s'].nunique()} rungs")
    display(Markdown("**Runs per town and rung** -- 6 is a complete cell "
                     "(2 policies x 3 seeds). Columns are request counts, "
                     "which is what a rung actually varies here."))
    display(frame.groupby(["town", "requests"]).size().unstack(fill_value=0))

## 4. Did the load actually land?

The same guard the sibling uses, and for the same reason: if occupancy does
not clear 60% this notebook has not reached the regime it was built to test
and section 6 cannot answer its question.

The impatient sibling reached **68.8%** on this ladder. Occupancy here should
land *higher* — the bus is carrying riders it would previously have lost to
abandonment — but that is a prediction, not an assumption, so it is measured.

In [ ]:
def saturation(runs: pd.DataFrame) -> pd.DataFrame:
    """
    Report how hard the bus was working at each rung.

    :param runs: Collected runs.
    :return: One row per town, rung and policy.
    """
    if runs.empty:
        return pd.DataFrame()
    rows = runs.groupby(["town", "offered_per_min", "policy"]).agg(
        sim_s=("sim_s", "mean"), throughput=("throughput", "mean"),
        ride_s=("ride", "mean"), fulfilled=("fulfilled", "mean"),
        n=("sim_s", "size")).reset_index()
    rows["drain_ratio"] = rows["sim_s"] / ARRIVAL_SPAN_S
    # Little's law: mean number aboard = arrival rate x time aboard.
    # throughput is fulfilled per minute, hence the /60 to reach per second.
    rows["seats_used"] = (100.0 * rows["throughput"] / 60.0 * rows["ride_s"]
                          / CAPACITY)
    rows["served_per_h"] = rows["fulfilled"] / (ARRIVAL_SPAN_S / 3600.0)
    return rows[["town", "offered_per_min", "policy", "drain_ratio",
                 "seats_used", "served_per_h", "n"]]


if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the sweep has run._"))
else:
    guard = saturation(frame)

    display(Markdown(
        f"**Seats used (%)** against {CAPACITY}. The impatient sibling "
        f"peaked at 68.8% on this ladder; this one should land higher, "
        "because the bus is now carrying riders it used to lose."))
    display(guard.pivot_table(index=["town", "policy"],
                              columns="offered_per_min",
                              values="seats_used").round(1))

    display(Markdown("**Drain ratio** -- simulated time over the "
                     f"{ARRIVAL_SPAN_S:.0f} s arrival span. Unbounded here "
                     "in principle; compare against the sibling's 1.2-1.5."))
    display(guard.pivot_table(index=["town", "policy"],
                              columns="offered_per_min",
                              values="drain_ratio").round(2))

    display(Markdown("**Fulfilled per hour of arrival span** -- the demand "
                     "actually absorbed."))
    display(guard.pivot_table(index=["town", "policy"],
                              columns="offered_per_min",
                              values="served_per_h").round(1))

    peak = guard["seats_used"].max()
    worst = guard.loc[guard["seats_used"].idxmax()]
    print(f"busiest cell: {worst['town']} {worst['policy']} at "
          f"{worst['offered_per_min']:.2f}/min -- "
          f"{peak:.1f}% of {CAPACITY} seats")
    if peak < INVERSION_OCCUPANCY:
        display(Markdown(
            f"> **The ladder did not reach the regime.** Peak occupancy "
            f"{peak:.1f}% is below the {INVERSION_OCCUPANCY:.0f}% the "
            "inversion needs, so section 6 cannot answer its question."))
    else:
        display(Markdown(
            f"> **The ladder reached the regime.** Peak occupancy "
            f"{peak:.1f}% is above {INVERSION_OCCUPANCY:.0f}%, so section 6 "
            "is testing the inversion rather than assuming it."))

## 5. Did every run actually finish?

**This section gates every number below it.** With no give-up, a finished
trial serves 100% of its requests. Anything less means the run hit `MAX_TIME`
with riders still pending, and a truncated run is not a slow run reported
honestly — its mean wait is computed over whoever happened to be delivered
before the clock stopped, which is precisely the censoring this notebook
exists to escape.

A truncated cell is dropped, not caveated.

In [ ]:
# A run that used all but this much of its clock was still working when it
# stopped. Not a round 100% because the runner overshoots max_time by at
# most one tick, and float ticks make an exact comparison brittle.
FINISHED_MARGIN_S = 60.0

if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the sweep has run._"))
    valid = frame
else:
    frame["truncated"] = (
        (frame["served"] < 99.9)
        | (frame["sim_s"] >= MAX_TIME - FINISHED_MARGIN_S))
    display(Markdown(
        "**Runs that did not finish.** With no give-up, coverage below "
        "100% means riders were still pending when the clock stopped."))
    display(frame.groupby(["town", "policy", "offered_per_min"]).agg(
        runs=("truncated", "size"), truncated=("truncated", "sum"),
        worst_served=("served", "min"),
        longest_sim_s=("sim_s", "max")).round(1))

    valid = frame[~frame["truncated"]].copy()
    lost = len(frame) - len(valid)
    if lost:
        display(Markdown(
            f"> **{lost} of {len(frame)} runs truncated and are dropped.** "
            "Their mean wait is computed over whoever happened to be "
            "delivered before `MAX_TIME`, which is the censoring this "
            "notebook exists to escape -- averaging them in would "
            "reintroduce it. Check section 8: a truncated run with an "
            "unroutable leg is a stuck bus, not a busy one."))
        # A cell that has lost some of its seeds is worse than one that has
        # lost all of them: the survivors are the easy seeds.
        complete = valid.groupby(
            ["town", "policy", "offered_per_min"]).size()
        partial = complete[complete < len(SEEDS)]
        if not partial.empty:
            display(Markdown(
                "> **These cells are now partial** -- their survivors are "
                "the seeds that happened to finish, so their means are "
                "biased toward the easy draws. Read them as anecdotes."))
            display(partial.rename("seeds_left").to_frame())
    else:
        display(Markdown(
            f"> **Every run finished.** All {len(frame)} served 100% "
            f"within {frame['sim_s'].max():.0f} s of the "
            f"{MAX_TIME:.0f} s ceiling, so wait time below is uncensored: "
            "every rider is in the mean."))

## 6. The test

Wait time against offered load and against measured occupancy, per town.

**Read the right panel.** The left one is the familiar view; the right is the
actual test, plotting routing's advantage against how full the bus was, with
the 60% line drawn where the synthetic map put the crossing.

Sign convention: **advantage = fixed route's wait minus Reordering's**, so
positive means routing is winning and a crossing below zero is the inversion.
Ride time and throughput are reported beside it, because the synthetic
inversion showed up on all three and a reversal on one alone would be a
weaker claim than a reversal on all of them.

In [ ]:
def advantage(runs: pd.DataFrame, column: str,
              lower_is_better: bool = True) -> pd.DataFrame:
    """
    Routing's advantage over fixed route, per town and rung.

    :param runs: Valid runs.
    :param column: KPI column to difference.
    :param lower_is_better: True for wait and ride, False for throughput.
        Sets the sign so that positive always means routing is winning.
    :return: Columns town, offered_per_min, seats_used, advantage.
    """
    if runs.empty:
        return pd.DataFrame()
    means = runs.groupby(["town", "offered_per_min", "policy"])[
        column].mean().unstack("policy")
    if not set(POLICY_LABELS) <= set(means.columns):
        return pd.DataFrame()
    gap = means["Fixed route"] - means["Reordering W750"]
    if not lower_is_better:
        gap = -gap
    out = gap.rename("advantage").reset_index()
    # Occupancy of the busier arm at each rung: the inversion is about how
    # full the bus is, and averaging both arms would blur a difference that
    # is itself part of the story.
    seats = saturation(runs).groupby(["town", "offered_per_min"])[
        "seats_used"].max().rename("seats_used").reset_index()
    return out.merge(seats, on=["town", "offered_per_min"])


if not HAVE_RUNS or valid.empty:
    display(Markdown("_Nothing to report until the sweep has run._"))
else:
    wait_gap = advantage(valid, "wait")
    towns = [t for t in TOWNS if t in set(valid["town"])]

    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
    for town in towns:
        block = valid[valid["town"] == town]
        for label in POLICY_LABELS:
            series = block[block["policy"] == label].groupby(
                "offered_per_min")["wait"].mean().sort_index()
            if series.empty:
                continue
            axes[0].plot(series.index, series.values, marker="o",
                         markersize=6, lw=2.0,
                         color=POLICY_COLOUR[label],
                         ls="-" if town == towns[0] else "--",
                         label=f"{town} {label}")
        arm = wait_gap[wait_gap["town"] == town].sort_values("seats_used")
        axes[1].plot(arm["seats_used"], arm["advantage"], marker="o",
                     markersize=6, lw=1.8, label=town)

    axes[0].set_title("Mean wait against offered load")
    axes[0].set_xlabel("offered demand (req/min)")
    axes[0].set_ylabel("mean wait (s)")
    axes[0].set_xticks(sorted(valid["offered_per_min"].unique()))
    axes[0].grid(axis="y")
    axes[0].set_axisbelow(True)
    axes[0].legend(fontsize=7)

    axes[1].axhline(0, lw=1.1, color=INK_SOFT)
    axes[1].axvline(INVERSION_OCCUPANCY, ls="--", lw=1.1, color=INK_SOFT)
    axes[1].set_title("Routing's wait advantage against occupancy")
    axes[1].set_xlabel(f"seats used (% of {CAPACITY})")
    axes[1].set_ylabel("fixed route - Reordering (s)")
    axes[1].grid(axis="y")
    axes[1].set_axisbelow(True)
    axes[1].legend(fontsize=8)
    figure.tight_layout()
    plt.show()

    for column, label, lower in (("wait", "Mean wait (s)", True),
                                 ("ride", "Mean ride (s)", True),
                                 ("throughput", "Throughput (req/min)",
                                  False)):
        display(Markdown(f"**{label}**, per policy."))
        display(valid.pivot_table(index=["town", "policy"],
                                  columns="offered_per_min",
                                  values=column).round(2))
        gap = advantage(valid, column, lower_is_better=lower)
        display(Markdown(
            f"_Routing's advantage on {column} -- positive means "
            "Reordering is winning._"))
        display(gap.pivot_table(index="town", columns="offered_per_min",
                                values="advantage").round(2))

In [ ]:
if not HAVE_RUNS or valid.empty:
    display(Markdown("_Nothing to report until the sweep has run._"))
else:
    spread = valid.groupby(["town", "offered_per_min", "policy"])[
        "wait"].std().groupby(["town", "offered_per_min"]).max()
    lines = []
    for _, row in advantage(valid, "wait").iterrows():
        noise = spread.get((row["town"], row["offered_per_min"]),
                           float("nan"))
        decisive = (pd.notna(noise) and abs(row["advantage"]) > noise)
        lines.append({
            "town": row["town"],
            "offered_per_min": round(row["offered_per_min"], 2),
            "seats_used": round(row["seats_used"], 1),
            "wait_advantage_s": round(row["advantage"], 1),
            "winner": ("Reordering" if row["advantage"] > 0
                       else "Fixed route"),
            "seed_spread_s": round(noise, 1) if pd.notna(noise) else None,
            "decisive": "yes" if decisive else "inside noise"})
    verdict = pd.DataFrame(lines).set_index(["town", "offered_per_min"])
    display(Markdown("**Verdict per cell.** `decisive` compares the "
                     "advantage against the worse arm's own seed spread -- "
                     "**`inside noise` means the winner beside it is not "
                     "evidence.**"))
    display(verdict)

    inverted = verdict[(verdict["winner"] == "Fixed route")
                       & (verdict["decisive"] == "yes")]
    reached = verdict["seats_used"].max()
    if not inverted.empty:
        where = ", ".join(f"{t} at {r:.2f}/min ({s:.0f}% seats)"
                          for (t, r), s in
                          inverted["seats_used"].items())
        display(Markdown(
            f"> **The ranking inverts.** Fixed route wins decisively in "
            f"{len(inverted)} cell(s): {where}. The synthetic map's scope "
            "condition transfers to real geometry in the patient regime, "
            "and the impatient sweep's null was a property of the regime "
            "rather than of the map."))
    else:
        display(Markdown(
            f"> **No decisive inversion anywhere on this ladder**, up to "
            f"{reached:.1f}% occupancy. Together with the impatient "
            "sibling's null, that puts the scope condition on the "
            "synthetic map rather than on CARLA -- but only up to the "
            "occupancy actually reached, and at three seeds."))

## 7. Patient against impatient, rung for rung

The matched pair. Same towns, same capacity, same span, same rungs, same
seeds, same window — `max_wait_s` alone differs, so the gap between the two
columns is the regime and nothing else.

**Both sides are read out of the store by the same function**, differing
only in the patience they filter on. The sibling notebook's section 7 got
this wrong by transcribing figures measured at a different capacity into a
hardcoded dict; reading both from the store is what makes that mistake
impossible here rather than merely unlikely.

Coverage cannot be compared across the regimes — it is 1.0 by construction on
this side — so the comparison is on **occupancy and throughput**, which are
defined the same way in both.

In [ ]:
impatient = collect(IMPATIENT_PATIENCE)

if not HAVE_RUNS or valid.empty:
    display(Markdown("_Nothing to report until the sweep has run._"))
elif impatient.empty:
    display(Markdown(
        "> **The impatient sibling's runs are not in this store.** Run "
        "`carla-sustained-load-capacity-12.ipynb` section 2, or read that "
        "notebook's own tables instead. Nothing else here depends on it."))
else:
    both = pd.concat([
        valid.assign(regime="patient"),
        impatient[impatient["town"].isin(valid["town"].unique())]
        .assign(regime="impatient")])

    display(Markdown(
        f"**Seats used (%)** against {CAPACITY}, by regime. The patient "
        "column should be the higher one -- riders who would have "
        "abandoned are aboard instead."))
    seats = pd.concat([
        saturation(both[both["regime"] == r]).assign(regime=r)
        for r in ("impatient", "patient")])
    display(seats.pivot_table(index=["town", "offered_per_min"],
                              columns="regime",
                              values="seats_used").round(1))

    display(Markdown(
        "**Throughput (fulfilled/min)**, by regime and policy. Defined the "
        "same way on both sides, unlike coverage."))
    display(both.pivot_table(index=["town", "policy"],
                             columns=["regime", "offered_per_min"],
                             values="throughput").round(2))

    display(Markdown(
        "**Mean wait (s)**, by regime and policy. **These two columns do "
        "not measure the same thing** and are placed side by side only to "
        "show how far apart they are: the impatient column is computed "
        "over survivors, and at a 750 s window against 750 s patience "
        "nobody who waited long is in it. The patient column is every "
        "rider. A large gap is the censoring, not a service difference."))
    display(both.pivot_table(index=["town", "policy"],
                             columns=["regime", "offered_per_min"],
                             values="wait").round(0))

## 8. Health

A run with repaths was physically stuck at some point; a run with unroutable
legs asked for a leg `plan_route` could not build. Neither invalidates a run
on its own, but they matter more here than in the sibling: with no give-up,
a rider whose pickup the bus cannot reach never cancels and holds the trial
open until `MAX_TIME`. **A truncated run in section 5 and an unroutable leg
here are probably the same event**, and that is a driving failure wearing a
demand failure's clothes.

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the sweep has run._"))
else:
    display(Markdown("**Per town and rung.** `stuck_runs` is the count of "
                     "runs with at least one repath."))
    display(frame.groupby(["town", "offered_per_min"]).agg(
        runs=("repaths", "size"), repaths=("repaths", "sum"),
        unroutable=("unroutable", "sum"),
        stuck_runs=("repaths", lambda c: int((c > 0).sum()))))

    suspect = frame[(frame["repaths"] > 0) | (frame["unroutable"] > 0)]
    print(f"{len(suspect)} of {len(frame)} runs carry a repath or an "
          f"unroutable leg")
    if not suspect.empty:
        display(suspect.groupby(["town", "policy", "offered_per_min"])[
            ["repaths", "unroutable", "served", "sim_s"]].mean().round(1))

    # The question this section is actually being asked: are the runs that
    # failed to finish the same runs that could not drive?
    if "truncated" in frame and frame["truncated"].any():
        display(Markdown(
            "**Truncated runs against driving failures.** If the truncated "
            "runs carry more repaths or unroutable legs than the rest, "
            "section 5 dropped stuck buses rather than slow ones -- which "
            "is a defect to chase, not a load result."))
        display(frame.groupby("truncated")[
            ["repaths", "unroutable", "served", "sim_s"]].mean().round(1))

## What this notebook can and cannot conclude

**It can say whether the synthetic map's inversion appears on real geometry
in the regime where the synthetic map found it.** That is one gap in the
project's evidence and it is the gap this closes.

**It cannot say anything about service.** The patient regime is a
measurement device, not a scenario: no deployment has passengers who wait
indefinitely, and a mean wait of several thousand seconds is a statement
about a queue, not about a bus anyone would ride. Every service claim in the
project stays with the impatient sweep.

Three limits to quote alongside whatever comes out:

- **Three seeds.** The `decisive` column is the guard; a verdict marked
  `inside noise` is a statement about the seeds.
- **Up to the occupancy actually reached**, which section 4 reports. A null
  here rules the inversion out over the measured range and no further.
- **Two towns**, chosen as the extremes of the impatient sweep's spread.
  Town03 was dropped for cost, not for being uninformative.